# LSTM Model Visualization

This notebook demonstrates how to use the visualization tools to analyze model predictions and behavior.

In [ ]:
import sys
sys.path.insert(0, '../')

In [ ]:
import torch
import os
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display
from visualization import Visualization
from scipy.stats import spearmanr

Get visualization object

In [ ]:

visualization = Visualization(
    "../checkpoints/lstm_v4_host-smp-n217.crc.pitt.edu_version3/model_best_score=val_loss=0.1113.ckpt",
    "lstmv4vlt"
)

Test if sequence of subject is ok (correct if True, True, False, True, False, True)

In [ ]:
_, _, _, id1, id2, _, _ = visualization._get_predictions("val", "training")
ref_id1, ref_id2 = np.array(id1), np.array(id2)
print(ref_id1[::100])
_, _, _, id1, id2, _, _ = visualization._get_predictions("val", "block")
id1, id2 = np.array(id1), np.array(id2)
print(np.array_equal(ref_id1[::100], id1[::34]))
_, _, _, id1, id2, _, _ = visualization._get_predictions("val", "full")
id1, id2 = np.array(id1), np.array(id2)
print(np.array_equal(ref_id1[::100], id1))

In [ ]:
from datasets.TimeSeriesDataModule import TimeSeriesDataModule


module = TimeSeriesDataModule(subcortical=True, window_timeserie=1190, samples_per_file=1, rate_output=False, identifiers_output=False, batch_size=32, scores_output=True)
module.setup("fit")

scores = []  
for batch in module.train_dataset:
    x, y, score = batch
    scores.append(score)
print(torch.stack(scores, dim=0).mean(dim=0), torch.stack(scores, dim=0).std(dim=0))
scores = []  
for batch in module.val_dataset:
    x, y, score = batch
    scores.append(score)
print(torch.stack(scores, dim=0).mean(dim=0), torch.stack(scores, dim=0).std(dim=0))

In [ ]:
visualization.analyze_brain_attributions("test", threshold=None)

In [ ]:
from PIL import Image

def create_gif(image_paths, output_gif_path, duration=100):
    """
    Creates a GIF from a list of image paths.

    Args:
        image_paths (list): List of image file paths (e.g., ["frame1.png", "frame2.png"]).
        output_gif_path (str): Path to save the output GIF (e.g., "output.gif").
        duration (int): Delay between frames in milliseconds (e.g., 100 for 0.1 seconds).
    """
    images = [Image.open(image_path) for image_path in image_paths]
    images[0].save(
        output_gif_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,  # 0 means infinite loop
    )

for cdt in ["worry", "neutral", "reappraisal"]:
    image_paths = [f"../results/lstmv4vlt/maps/plot/{cdt}_map_time-{i}.png" for i in range(25)]  # Replace with your image paths
    output_gif_path = f"../results/lstmv4vlt/maps/plot/{cdt}.gif"
    create_gif(image_paths, output_gif_path, duration=500)  # 100ms delay
    print(f"GIF created and saved at {output_gif_path}")


# Metrics

## Visualisation

In [ ]:
for i in range(10):
    for step in ["val", "test"]:
        visualization.visualize_subject(i, step=step)

## Loss MAE

In [ ]:
for pop in ["val", "test"]:
    for t in ["training", "block", "full"]:
        print(visualization.loss_compute(pop, t))

## AUC

Validation using block

In [ ]:
m = visualization.visualize_roc_metrics_rates()

# Extract categories dynamically
categories = list(m[0].keys())

# Initialize a list to store DataFrames for each category
dataframes = {}

for category in categories:
    records = []
    for idx, metrics in enumerate(m):
        row = {'Dataset': visualization._comparisons[idx]}
        row.update(metrics[category])
        records.append(row)
    
    df = pd.DataFrame(records)
    dataframes[category] = df

for category, df in dataframes.items():
    print(f"\n### {category} ###\n")
    display(df)

Validation using full

In [ ]:
m = visualization.visualize_roc_metrics_rates("full")

# Extract categories dynamically
categories = list(m[0].keys())

# Initialize a list to store DataFrames for each category
dataframes = {}

for category in categories:
    records = []
    for idx, metrics in enumerate(m):
        row = {'Dataset': visualization._comparisons[idx]}
        row.update(metrics[category])
        records.append(row)
    
    df = pd.DataFrame(records)
    dataframes[category] = df

for category, df in dataframes.items():
    print(f"\n### {category} ###\n")
    display(df)

## Effect of delta rating

Get formatted data


In [ ]:
all_data = pd.concat([
   pd.DataFrame({**{'Rate': (i+int(not t)), 'Type':t, 'Selection': asc}, **metrics, 'Dataset': visualization._comparisons[idx]})
   for t in [True, False]
   for asc in [0, 1, 2]
   for i in range(5)
   for idx, metrics in enumerate(visualization.visualize_roc_metrics_rates(threshold=(i+int(not t)), asc=asc, delta=t))
])
all_data = all_data.reset_index(drop=False)
all_data = all_data.rename(columns={'index': 'Metrics'})
display(all_data)


Validation Delta and Rating

In [ ]:
print("---- Delta Rating ----")
for d in pd.unique(all_data['Dataset']):
    tmp = all_data[(all_data['Type']) & (all_data['Metrics'] == "auc") & (all_data['Dataset'] == d) & ((all_data['Selection'] == 2))]
    tmp.loc['Rate'] = tmp['Rate'].astype('category')
    tmp = tmp.groupby("Rate")[["validation", "test"]].mean().reset_index()
    
    stat1, pval1 = spearmanr(tmp["Rate"], tmp["validation"])
    stat1, pval1 = round(stat1, 3), pval1 < 0.05
    stat2, pval2 = spearmanr(tmp["Rate"], tmp["test"])
    stat2, pval2 = round(stat2, 3), pval2 < 0.05
    print(d, stat1, pval1, stat2, pval2)

print("---- Rating ----")
for d in pd.unique(all_data['Dataset']):
    tmp = all_data[(~all_data['Type']) & (all_data['Metrics'] == "auc") & (all_data['Dataset'] == d) & ((all_data['Selection'] == 2))]
    tmp.loc['Rate'] = tmp['Rate'].astype('category')
    tmp = tmp.groupby("Rate")[["validation", "test"]].mean().reset_index()
    stat1, pval1 = spearmanr(tmp["Rate"], tmp["validation"])
    stat1, pval1 = round(stat1, 3), pval1 < 0.05
    stat2, pval2 = spearmanr(tmp["Rate"], tmp["test"])
    stat2, pval2 = round(stat2, 3), pval2 < 0.05
    print(d, stat1, pval1, stat2, pval2)

Delta correct labels plot

In [ ]:
tmp = all_data[(all_data['Type']) & (all_data["Metrics"] == "auc") & (all_data["Selection"] == 2)]
tmp = pd.melt(tmp, id_vars=["Dataset", "Rate"], value_vars=['validation','test'], value_name='AUC', var_name="Set")

g = sns.catplot(
    data=tmp,
    x="Dataset", y="AUC", hue="Rate", kind="bar", col='Set',
    height=5, errorbar=('ci', 95)
)
g.set_xlabels("Conditions")
g.tick_params(axis='x', rotation=90)
g.legend.set_title('Worrying score evolution')
new_labels = ['No difference', '+/- 1', '+/- 2', '+/- 3', '+/- 4']
for t, l in zip(g.legend.texts, new_labels):
    t.set_text(l)
sns.move_legend(g, "upper right", bbox_to_anchor=(.9, .9))
plt.ylim(0.5, 1)
plt.show()

Rating correct labels plot

In [ ]:
tmp = all_data[(~all_data['Type']) & (all_data["Metrics"] == "auc") & (all_data["Selection"] == 2)]
tmp = pd.melt(tmp, id_vars=["Dataset", "Rate"], value_vars=['validation','test'], value_name='AUC', var_name="Set")

g = sns.catplot(
    data=tmp,
    x="Dataset", y="AUC", hue="Rate", kind="bar", col='Set',
    height=5, errorbar=('ci', 95)
)
g.set_xlabels("Conditions")
g.tick_params(axis='x', rotation=90)
g.legend.set_title('Worrying score')
new_labels = list(range(1,6))
for t, l in zip(g.legend.texts, new_labels):
    t.set_text(l)
sns.move_legend(g, "upper right", bbox_to_anchor=(.9, .9))
plt.ylim(0.5, 1)
plt.show()

In [ ]:
batchV = visualization._get_predictions("val", "block")
_, y_hatV, yV, id1V, _, ratesV = batchV
batchT = visualization._get_predictions("test", "block")
_, y_hatT, yT, id1T, _, ratesT = batchT
y_hat = torch.cat((y_hatV, y_hatT), dim=0)
y = torch.cat((yV, yT), dim=0)
rates = np.concatenate((np.array(ratesV), np.array(ratesT)), axis=0)
id1 = np.concatenate((np.array(id1V), np.array(id1T)), axis=0) if type == "block" else np.concatenate((np.array(id1V), np.array(id1T)), axis=0)

print(visualization._calculate_one_vs_rest_metrics(y[:, 0, :], y_hat[:, 0, :], 0))

# Concatenate all DataFrames into one
all_data = pd.concat([
    pd.DataFrame({
        'Timing': t, 
        'Label': visualization._comparisons[l], 
        **(visualization._calculate_one_vs_rest_metrics(yV[:, t, :], y_hatV[:, t, :], l) if l < 3 else visualization._calculate_one_vs_one_metrics(y[:, t, :], y_hat[:, t, :], visualization.contrasts[l-3][0], visualization.contrasts[l-3][1]))
    }, index=range(75))
    for t in range(25)
    for l in range(6)
])

all_data = all_data.reset_index(drop=False)
display(all_data)
display(all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index())
sns.lineplot(x="Timing", y="auc",
             hue="Label",
             data=all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index())

display(all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index().groupby("Label")["auc"].idxmax().to_numpy() - np.arange(6)*25)

all_data = pd.concat([
    pd.DataFrame({
        'Timing': t, 
        'Label': visualization._comparisons[l], 
        **(visualization._calculate_one_vs_rest_metrics(yT[:, t, :], y_hatT[:, t, :], l) if l < 3 else visualization._calculate_one_vs_one_metrics(y[:, t, :], y_hat[:, t, :], visualization.contrasts[l-3][0], visualization.contrasts[l-3][1]))
    }, index=range(75))
    for t in range(25)
    for l in range(6)
])

all_data = all_data.reset_index(drop=False)
display(all_data)
display(all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index())
display(all_data.groupby(["Label"])["auc"].mean())

tmp = all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index()
tmp["auc_scaled"] = tmp.groupby("Label")["auc"].transform(lambda x: (x - x.min())/(x.max()-x.min()))
sns.lineplot(x="Timing", y="auc_scaled",
             hue="Label",
             data=tmp)

display(all_data.groupby(["Label", "Timing"])["auc"].mean().reset_index().groupby("Label")["auc"].idxmax().to_numpy() - np.arange(6)*25)

## Heatmap

In [ ]:
a = visualization.generate_heatmap("val", 5, delta_rating=True)
b = visualization.generate_heatmap("test", 5, delta_rating=True)

# Explainer

## Heatmap & Stream

In [ ]:
for pop in ["val", "test"]:
    for label in range(3):
        visualization.generate_network_heatmap(pop, label, "gaussian", use_network=True, only_concerned_block=True)
    for diff in [[0,1], [0,2], [1,2]]:
        visualization.generate_diffnetwork_heatmap(pop, diff, "gaussian", only_concerned_block=True)       

Correlations between clinical and networks

In [ ]:
from scipy.special import softmax
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

network = pd.read_csv("../external/schaeffer_subcortical_smith_templates.csv")
data, prediction, labels, id1, id2, rate, scores = visualization._get_predictions("test", "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
scores = np.array([score.numpy() for score in scores])
print(scores.shape)
scoresM = np.median(scores.reshape(88, 34, 3), axis=1)

weights = network.iloc[:, 1:].to_numpy()
weights = weights / np.sqrt(np.sum(weights ** 2, axis=1, keepdims=True))

# Shap value
shap_vals = visualization.shap_test
shap_vals = shap_vals.reshape(88, 34, 25, 481, 3)
shap_norm = shap_vals / np.sqrt(np.sum(shap_vals ** 2, axis=3, keepdims=True))
shap_network = np.matmul(shap_norm.reshape(-1, 481), weights.T).reshape(88*34, 25, 18, 3)
print(shap_vals.shape)

# Shap value
x = data.detach().numpy()
x = x.reshape(88, 34, 25, 481)
x_norm = x / np.sqrt(np.sum(x ** 2, axis=3, keepdims=True))
x_network = np.matmul(x_norm.reshape(-1, 481), weights.T).reshape(88*34, 25, 18)
print(x.shape)

# Correlation
for l in range(3):
    corr = np.zeros((18, 3))
    mask = np.argmax(softmax(labels.numpy().sum(1), axis=-1), axis=-1) == l
    for s in range(3):
        for i in range(18):
            tmp = np.abs(shap_network)[mask, :, i, l].reshape(88, [16, 10, 8][l], 25).mean(axis=(1, 2))
            corr[i, s] = spearmanr(tmp, scoresM[:, s]).statistic
    data = pd.DataFrame(corr.T, columns=network.iloc[:, 0])
    data.to_csv(f'../results/lstmv4vlt/shap-clinicsxnetworks-corr_label-{visualization._labels[l]}_pop-test.csv')
for l in range(3):
    corr = np.zeros((18, 3))
    mask = np.argmax(softmax(labels.numpy().sum(1), axis=-1), axis=-1) == l
    for s in range(3):
        for i in range(18):
            tmp = np.abs(x_network)[mask, :, i].reshape(88, [16, 10, 8][l], 25).mean(axis=(1, 2))
            corr[i, s] = spearmanr(tmp, scoresM[:, s]).statistic
    data = pd.DataFrame(corr.T, columns=network.iloc[:, 0])
    data.to_csv(f'../results/lstmv4vlt/x-clinicsxnetworks-corr_label-{visualization._labels[l]}_pop-test.csv')
data

Time effect

In [ ]:
shap = np.abs(visualization.shap_vals).mean(axis=(0, 2))
shap = (shap - shap.min(axis=0, keepdims=True))/(shap.max(axis=0, keepdims=True) - shap.min(axis=0, keepdims=True))
plt.plot(shap)
plt.legend(visualization._labels)
plt.show()
print("Seconds:", np.argpartition(shap[:, 0], -4)[-4:]+1)
print("Seconds:", np.argpartition(shap[:, 1], -4)[-4:]+1)
print("Seconds:", np.argpartition(shap[:, 2], -4)[-4:]+1)

In [ ]:
shap = np.abs(visualization.shap_test).mean(axis=(0, 2))
shap = (shap - shap.min(axis=0, keepdims=True))/(shap.max(axis=0, keepdims=True) - shap.min(axis=0, keepdims=True))
plt.plot(shap)
plt.legend(visualization._labels)
plt.show()
print("Seconds:", np.argpartition(shap[:, 0], -4)[-4:]+1)
print("Seconds:", np.argpartition(shap[:, 1], -4)[-4:]+1)
print("Seconds:", np.argpartition(shap[:, 2], -4)[-4:]+1)

## Clinical data

In [ ]:
fina = pd.read_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "FINA", "Public", "Analysis", "misc", "FINA_2025_01_17.csv"))
print(", ".join(fina.columns))
cols = ["pswq_total", "rsq_total", "hars_score", "age", "sex", "madrs_total", "education"]
#["imis", "vcis", "lis", "ais", "mdmis", "mtotalis"] + ["pswq_total", "rsq_total", "hars_score", "pss_score", "age", "gender", "race", "cirstot", "hamd_total", "neoffi_score_e1", "neoffi_score_o1", "neoffi_score_a1", "neoffi_score_c1"]
#['psqi_global', 'phq9_score', 'gad7_total', 'mmtotal']
fina['hamd_total'] = pd.NA
fina["age"] = fina["age_at_mrscan"]
fina["gender"] = fina["sex"]
fina["Vault_UID"] = fina["Vault_ScanID"]
data, prediction, labels, id1, id2, rate, m = visualization._get_predictions("train", "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
df = fina[fina.Vault_UID.isin(id2)]
df = df[["Vault_UID"]+cols]
df.to_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "training.csv"), index=False)
df.describe()

In [ ]:
fina = pd.read_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "FINA", "Public", "Analysis", "misc", "FINA_2025_01_17.csv"))
print(", ".join(fina.columns))
cols = ["pswq_total", "rsq_total", "hars_score", "age", "sex", "madrs_total", "education", "hamd_total"]
fina['hamd_total'] = pd.NA
fina["age"] = fina["age_at_mrscan"]
fina["gender"] = fina["sex"]
fina["Vault_UID"] = fina["Vault_ScanID"]
data, prediction, labels, id1, id2, rate, m = visualization._get_predictions("val", "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
df = fina[fina.Vault_UID.isin(id2)]
df = df[["Vault_UID"]+cols]
df.to_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "val.csv"), index=False)
df.describe()

In [ ]:
raw = pd.read_csv(os.path.join("/mnt/argo/Workspaces", "Staff", "Jacques-Yves_Campion", "Public", "Project-1_WorryInduction", "Data", "RAW_JY_Data_Request_2.10.25.csv"))
data, prediction, labels, id1, id2, rate, memories = visualization._get_predictions("test", "block")
id1, id2, rate = np.array(id1), np.array(id2), np.array(rate)
raw["Vault_UID"] = raw.raw_id.str.replace("RAW_", "").astype(np.int64)
df = raw[raw.Vault_UID.isin(id2.astype(np.int64))]
print(", ".join(df.columns))
df['sex'] = df['gender']
df['madrs_total'] = pd.NA
cols = ["pswq_total", "rsq_total", "hars_score", "age", "sex", "madrs_total", "hamd_total", "raw_id"]
df = df[cols]
print(len(df[df.sex==3]))
df.to_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "test.csv"), index=False)
df.describe()

In [ ]:
educ = pd.read_csv(os.path.join("/mnt/argo/Workspaces", "Staff", "Jacques-Yves_Campion", "Public", "Project-1_WorryInduction", "Data", "pheno_raw.csv"))
educ = educ[['raw_id', 'education']]
df = df.merge(educ, how="left", on="raw_id")
print(", ".join(df.columns))
df = df[cols+['education']]
df.to_csv(os.path.join(visualization.full_dataset_trainvaltest.base_dir, "test.csv"), index=False)
df.describe()